In [0]:
%run ../../config/utils

In [0]:
import sys
sys.path.append("..")
sys.path.append("../..")

from lib.job_manager import load_config, split_config
from lib_etl.s3 import etl_input_data_validator
import lib_etl.validations_ETL as validations

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)

## Data availability check

In [0]:
source_path = data_paths["source"]["club"]
recency_lookback_duration = data_paths.get("recency_lookback_duration", {})
etl_input_data_validator(
    "source",
    recency_lookback_duration,
    data_paths,
    [
        "club",
    ],
    spark
)

## Load source data

In [0]:
df = spark.read.csv(source_path, header=True, sep="|")
df.createOrReplaceTempView('df')

In [0]:
validations.validate_table(
        spark, "source", 'club', config_validation, df, stats_etl_path
    )

## Save to delta table

In [0]:
# spark.sql(f"""
#     INSERT OVERWRITE {bronze_master_club_with_brand}
#     SELECT
#         _c0,
#         _c1,
#         _c2,
#         _c3,
#         _c4,
#         _c5,
#         _c6,
#         _c7,
#         _c8,
#         _c9
#     FROM df
# """)

In [0]:
# if you get to use the file header you can update the table definition and use:

spark.sql(f"""
    INSERT OVERWRITE {bronze_master_club_with_brand}
    SELECT
        SITE_NBR,
        SITE_NAME_2,
        ADDR_LINE_2,
        CITY_NAME,
        STATE_CD,
        ZIP_CD,
        ZN_NBR,
        RGN_NBR,
        SITE_TYPE,
        COMP_STTS
    FROM df
""")